# FlyGraph Attention vs SmolLM2-135M — corrected experiment

The previous notebook used a fixed recurrent FlyWire message sum. That was **not attention** because the connection strength did not depend on the current query/key content.

This corrected experiment replaces **every SmolLM2-135M self-attention layer** with **FlyGraph causal linear attention**. It keeps SmolLM2's pretrained embeddings, RMSNorm, MLP/residual stack, and initializes Q/K/V/O from the pretrained model, but removes standard `LlamaAttention`.

The replacement performs real Q/K/V content retrieval with constant-size causal statistics:

`S_t = S_(t-1) + phi_fly(k_t) v_t^T`

`Z_t = Z_(t-1) + phi_fly(k_t)`

`y_t = phi_fly(q_t)^T S_t / (phi_fly(q_t)^T Z_t)`

`phi_fly` is a positive softmax-kernel feature map mixed over a real FlyWire v783 subgraph with content-dependent edge gating. A rewired graph is trained as a control.

The script also asserts that the final student contains **0 standard LlamaAttention modules** and checks that full-sequence and streaming decoding agree numerically.

In [ ]:
#@title 1. Clone/update repository and install
import pathlib, subprocess, sys

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR),
    "transformers>=4.56", "datasets>=3.0", "accelerate>=1.0",
    "huggingface_hub>=0.34", "pandas>=2.0", "requests>=2.31", "tqdm>=4.66"
], check=True)

print("Repository ready:", REPO_DIR)


In [ ]:
#@title 2. Choose experiment size
RUN_MODE = "quick"  #@param ["quick", "strong"]
SEQ_LEN = 128       #@param {type:"integer"}
FEATURE_DIM = 256   #@param {type:"integer"}
MAX_EDGES = 2048    #@param {type:"integer"}
GRAPH_STEPS = 1     #@param {type:"integer"}
RUN_REWIRED_CONTROL = True  #@param {type:"boolean"}

OUTPUT_DIR = REPO_DIR / "results" / "flygraph_attention_smollm2_135m"
print("Output:", OUTPUT_DIR)


In [ ]:
#@title 3. Train/evaluate corrected attention replacement
cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "run_smollm2_fly_graph_attention.py"),
    "--run-mode", RUN_MODE,
    "--seq-len", str(SEQ_LEN),
    "--feature-dim", str(FEATURE_DIM),
    "--max-edges", str(MAX_EDGES),
    "--graph-steps", str(GRAPH_STEPS),
    "--output-dir", str(OUTPUT_DIR),
]
if RUN_REWIRED_CONTROL:
    cmd.append("--rewired")

print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 4. Show results
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(OUTPUT_DIR / "summary.csv", index_col=0)
report = json.loads((OUTPUT_DIR / "report.json").read_text())

display(summary)

print("\nATTENTION CHECK")
print("pure_attention_replacement:", report["pure_attention_replacement"])
print("standard_llama_attention_modules:", report["standard_llama_attention_modules"])
print("streaming_full_max_abs_logit_diff:", report["streaming_full_max_abs_logit_diff"])

print("\nKEY METRICS")
for key in (
    "fly_ce_gap_vs_smollm2",
    "fly_ppl_ratio_vs_smollm2",
    "decode_speed_ratio_fly_over_smollm2",
    "biological_topology_ce_gain",
    "biological_topology_ppl_gain_pct",
):
    if key in report:
        print(f"{key}: {report[key]}")

print("\nHow to read this:")
print("1) CE/PPL should be dramatically better than the old FlyCeNN CE≈8.24 / PPL≈3781.")
print("2) pure_attention_replacement must be True and standard_llama_attention_modules must be 0.")
print("3) streaming_full_max_abs_logit_diff should be tiny; otherwise streaming decode is incorrect.")
print("4) biological_topology_ce_gain > 0 means real FlyWire wiring beat the rewired control.")
print("5) If quick mode closes much of the SmolLM2 gap, rerun with RUN_MODE='strong'.")
